# ☀️ Solproduktionsprognos — SMHI STRÅNG + ML
**Steg-för-steg pipeline:** STRÅNG historik → feature engineering → modellträning → rapport → styrning

## Cell 1 — Imports & konstanter

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from io import StringIO
from datetime import datetime, timedelta
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Konstanter ──
LAT, LON      = 56.67, 12.86          # Halmstad
STATION_ID    = 62410                  # Halmstad Flygplats
MASTER_CSV    = "Datainsamling/master_energy_data_cleaned.csv"
TRAIN_CUTOFF  = "2025-07-01"          # Allt före = träning
STRANG_PARAM  = 117                    # Global irradians W/m²


✅ Imports klara


## Cell 2 — Ladda master CSV

In [ ]:
df_master = pd.read_csv(MASTER_CSV, parse_dates=['timestamp'])
df_master = df_master.sort_values('timestamp').set_index('timestamp')
df_master.index = df_master.index.tz_localize('UTC').tz_convert('Europe/Stockholm')

print(f"Rader:   {len(df_master):,}")
print(f"Kolumner: {df_master.columns.tolist()}")
print(f"Period:  {df_master.index.min()} → {df_master.index.max()}")
df_master.head(3)


## Cell 3 — Hämta STRÅNG historisk GHI (Halmstad)

In [3]:
def fetch_strang_hourly(lat, lon, param, date_from, date_to):
    """
    Hämtar timvis strålningsdata från SMHI STRÅNG.
    Returnerar en pd.Series med DateTime-index (Europe/Stockholm).
    Saknade värden (-999) ersätts med NaN.
    """
    from_str = date_from.strftime('%Y%m%d%H')   # format: YYYYMMDDHH
    to_str   = date_to.strftime('%Y%m%d%H')

    url = (
        f"https://opendata-download-metanalys.smhi.se/api/category/strang1g"
        f"/version/1/geotype/point/lon/{lon}/lat/{lat}"
        f"/parameter/{param}/data.txt"
        f"?from={from_str}&to={to_str}&interval=hourly"
    )
    print(f"Hämtar: {url}")
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    # Parsa ASCII: år mån dag timme värde
    rows = []
    for line in r.text.strip().splitlines():
        parts = line.split()
        if len(parts) == 5:
            try:
                ts = pd.Timestamp(year=int(parts[0]), month=int(parts[1]),
                  day=int(parts[2]), hour=int(parts[3]),
                  tz='UTC'
                ).tz_convert('Europe/Stockholm').tz_localize(None)
                val = float(parts[4])
                rows.append((ts, val))
            except ValueError:
                continue

    s = pd.Series(
        {ts: v for ts, v in rows},
        name='strang_ghi_wm2'
    )
    s = s.replace(-999, np.nan)   # Saknade värden → NaN
    print(f"✅ STRÅNG: {len(s):,} timmar | "
          f"Saknade: {s.isna().sum()} | "
          f"Max: {s.max():.0f} W/m²")
    return s

# Hämta hela perioden
date_from = datetime(2023, 7, 1, 0)
date_to   = datetime(2025, 12, 31, 23)
strang_series = fetch_strang_hourly(LAT, LON, STRANG_PARAM, date_from, date_to)
strang_series.head()


Hämtar: https://opendata-download-metanalys.smhi.se/api/category/strang1g/version/1/geotype/point/lon/12.86/lat/56.67/parameter/117/data.txt?from=2023070100&to=2025123123&interval=hourly
✅ STRÅNG: 21,957 timmar | Saknade: 192 | Max: 871 W/m²


2023-07-01 02:00:00    0.000
2023-07-01 03:00:00    0.000
2023-07-01 04:00:00    0.000
2023-07-01 05:00:00    8.200
2023-07-01 06:00:00   54.300
Name: strang_ghi_wm2, dtype: float64

## Cell 4 — Hämta SMHI metobs — temperatur (station 62410)

In [ ]:
def fetch_metobs_temperature(station_id):
    """
    Hämtar timvis lufttemperatur (parameter 1) från SMHI metobs.
    Kombinerar corrected-archive + latest-months automatiskt.
    Returnerar en pd.Series med DateTime-index (UTC→Europe/Stockholm).
    """
    base = "https://opendata-download-metobs.smhi.se/api/version/1.0"
    frames = []

    for period in ['corrected-archive', 'latest-months']:
        url = f"{base}/parameter/1/station/{station_id}/period/{period}/data.csv"
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            # SMHI CSV har metadata-rader i toppen — hoppa över dem
            lines = r.content.decode('utf-8-sig').splitlines()
            data_start = next(
                i for i, l in enumerate(lines)
                if l.startswith('Datum') or l.startswith('datum')
            )
            csv_text = '
'.join(lines[data_start:])
            tmp = pd.read_csv(
                StringIO(csv_text),
                sep=';',
                usecols=[0, 1, 2],
                names=['datum', 'tid_utc', 'temp_obs'],
                skiprows=1,
                dtype={'temp_obs': str}
            )
            tmp['timestamp'] = pd.to_datetime(
                tmp['datum'] + ' ' + tmp['tid_utc'],
                utc=True,           # SMHI kör UTC av någon anledning
                errors='coerce'
            ).dt.tz_convert('Europe/Stockholm').dt.tz_localize(None)  # → lokal tid
            tmp['temp_obs'] = pd.to_numeric(tmp['temp_obs'], errors='coerce')
            frames.append(tmp[['timestamp', 'temp_obs']].dropna())
            print(f"  Period '{period}': {len(tmp)} rader")
        except Exception as e:
            print(f"  Period '{period}': hoppades över ({e})")

    df_obs = pd.concat(frames).drop_duplicates('timestamp')
    df_obs = df_obs.set_index('timestamp').sort_index()['temp_obs']
    df_obs.name = 'temp_obs_c'
    print(f"\n✅ metobs temp: {len(df_obs):,} timmar | "
          f"Period: {df_obs.index.min().date()} → {df_obs.index.max().date()}")
    return df_obs

temp_obs = fetch_metobs_temperature(STATION_ID)
temp_obs.head()


## Cell 5 — Merge & feature engineering

In [ ]:
# Slå ihop alla datakällor
df = df_master.copy()
df['strang_ghi']  = strang_series
df['temp_obs']    = temp_obs

# Interpolera korta luckor i STRÅNG (max 2h)
df['strang_ghi'] = df['strang_ghi'].interpolate(method='time', limit=2)

# ── Kalenderfeatures ──
df['hour']       = df.index.hour
df['month']      = df.index.month
df['weekday']    = df.index.dayofweek
df['day_of_year']= df.index.dayofyear

# ── Solhöjdsapproximation (enkel, utan ephem) ──
# Solhorisontell strålning minskar mot morgon/kväll → modelleras med sin-kurva
df['solar_angle'] = np.sin(
    np.pi * (df['hour'] - 6) / 14          # 0 vid 6:00, max vid 13:00, 0 vid 20:00
).clip(lower=0)

# ── Daglig aggregat-features (för prognos dagen efter) ──
daily = df.resample('D').agg(
    production     = ('total_production_kwh', 'sum'),
    strang_mean    = ('strang_ghi', 'mean'),
    strang_max     = ('strang_ghi', 'max'),
    temp_max       = ('temp_obs', 'max'),
    temp_mean      = ('temp_obs', 'mean'),
    house_load_sum = ('total_house_load_kwh', 'sum'),
    month          = ('month', 'first'),
    weekday        = ('weekday', 'first'),
    day_of_year    = ('day_of_year', 'first'),
).dropna(subset=['production', 'strang_mean'])

# ── Laggade features (ingen data leakage!) ──
for lag in [1, 2, 7]:
    daily[f'prod_lag{lag}']   = daily['production'].shift(lag)
    daily[f'strang_lag{lag}'] = daily['strang_mean'].shift(lag)

daily['prod_roll7']   = daily['production'].shift(1).rolling(7).mean()
daily['prod_roll14']  = daily['production'].shift(1).rolling(14).mean()
daily['strang_roll7'] = daily['strang_mean'].shift(1).rolling(7).mean()

# ── Årstidscykel ──
daily['season_sin'] = np.sin(2 * np.pi * daily['day_of_year'] / 365.25)
daily['season_cos'] = np.cos(2 * np.pi * daily['day_of_year'] / 365.25)

# Target = nästa dags produktion
daily['target'] = daily['production'].shift(-1)
daily = daily.dropna()

print(f"✅ Datamängd redo: {len(daily)} dagar")
print(f"   Träning (< {TRAIN_CUTOFF}): {(daily.index < TRAIN_CUTOFF).sum()} dagar")
print(f"   Test   (≥ {TRAIN_CUTOFF}): {(daily.index >= TRAIN_CUTOFF).sum()} dagar")
daily.head(3)


## Cell 6 — Träna modell (HistGradientBoosting)

In [ ]:
FEATURES = [
    'strang_lag1', 'strang_lag2', 'strang_lag7', 'strang_roll7',
    'strang_max',
    'temp_max', 'temp_mean',
    'prod_lag1', 'prod_lag2', 'prod_lag7',
    'prod_roll7', 'prod_roll14',
    'month', 'weekday', 'season_sin', 'season_cos',
]

# Temporal split
mask_train = daily.index < TRAIN_CUTOFF
mask_test  = daily.index >= TRAIN_CUTOFF

X_train = daily.loc[mask_train, FEATURES]
y_train = daily.loc[mask_train, 'target']
X_test  = daily.loc[mask_test, FEATURES]
y_test  = daily.loc[mask_test, 'target']

# HistGradientBoosting = sklearn-native, hanterar NaN, snabb
model = HistGradientBoostingRegressor(
    max_iter        = 500,
    learning_rate   = 0.05,
    max_depth       = 5,
    min_samples_leaf= 10,
    l2_regularization= 0.1,
    random_state    = 42,
    early_stopping  = True,
    validation_fraction= 0.1,
    n_iter_no_change= 30,
)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2   = r2_score(y_true, y_pred)
    print(f"{label:10}  MAE={mae:.2f} kWh  RMSE={rmse:.2f} kWh  R²={r2:.4f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

print(f"Antal träningsdagar: {len(X_train)}")
print(f"Tidiga stopp vid iteration: {model.n_iter_}")
print()
res_train = metrics(y_train, y_pred_train, "TRÄNING")
res_test  = metrics(y_test,  y_pred_test,  "TEST")


## Cell 7 — Rapport & visualiseringar

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Solproduktionsprognos — Modellrapport', fontsize=14, fontweight='bold')

# ── Panel 1: Prognos vs faktisk (testperiod) ──
ax = axes[0, 0]
ax.plot(daily.loc[mask_test].index, y_test.values,
        label='Faktisk', color='#01696f', linewidth=1.5, alpha=0.85)
ax.plot(daily.loc[mask_test].index, y_pred_test,
        label='Prognos', color='#da7101', linewidth=1.5, alpha=0.85, linestyle='--')
ax.set_title('Testperiod: Prognos vs Faktisk')
ax.set_ylabel('kWh/dag')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend()
ax.grid(alpha=0.3)

# ── Panel 2: Scatter prognos vs faktisk ──
ax = axes[0, 1]
ax.scatter(y_test, y_pred_test, alpha=0.5, color='#01696f', s=20)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, 'r--', linewidth=1)
ax.set_xlabel('Faktisk (kWh)'); ax.set_ylabel('Prognos (kWh)')
ax.set_title(f"Scatter  R²={res_test['r2']:.3f}  MAE={res_test['mae']:.1f} kWh")
ax.grid(alpha=0.3)

# ── Panel 3: Residualfördelning ──
ax = axes[1, 0]
residuals = y_pred_test - y_test.values
ax.hist(residuals, bins=40, color='#006494', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--')
ax.axvline(residuals.mean(), color='orange', linestyle='--',
           label=f'Medel: {residuals.mean():.1f} kWh')
ax.set_xlabel('Residual (kWh)'); ax.set_ylabel('Antal dagar')
ax.set_title('Residualfördelning (test)')
ax.legend(); ax.grid(alpha=0.3)

# ── Panel 4: Feature importance ──
ax = axes[1, 1]
imp_result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
imp_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': imp_result.importances_mean
}).sort_values('importance', ascending=True).tail(12)
bars = ax.barh(imp_df['feature'], imp_df['importance'], color='#01696f', alpha=0.85)
ax.set_xlabel('Permutation Importance (↑ viktigare)')
ax.set_title('Feature Importance (permutation, testset)')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('model_report.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Rapport sparad: model_report.png")


## Cell 8 — Hämta STRÅNG + SMHI pmp3g — morgondagensprognos

In [ ]:
def fetch_strang_forecast_day(lat, lon, param, target_date):
    """
    Hämtar STRÅNG-data för ett specifikt dygn.
    Notera: STRÅNG är en analysmodell, inte prognos.
    Den senaste tillgängliga dagen används som proxy.
    För riktig prognos, kombinera med pmp3g-molntäckning (se nedan).
    """
    date_str_from = target_date.strftime('%Y%m%d00')
    date_str_to   = target_date.strftime('%Y%m%d23')
    url = (
        f"https://opendata-download-metanalys.smhi.se/api/category/strang1g"
        f"/version/1/geotype/point/lon/{lon}/lat/{lat}"
        f"/parameter/{param}/data.txt"
        f"?from={date_str_from}&to={date_str_to}&interval=hourly"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    rows = {}
    for line in r.text.strip().splitlines():
        p = line.split()
        if len(p) == 5:
            try:
                ts = pd.Timestamp(year=int(p[0]), month=int(p[1]),
                                  day=int(p[2]), hour=int(p[3]), tz='UTC')
                rows[ts.tz_convert('Europe/Stockholm')] = float(p[4])
            except ValueError:
                pass
    return pd.Series(rows, name='strang_ghi_forecast').replace(-999, np.nan)


def fetch_pmp3g_forecast(lat, lon):
    """
    Hämtar SMHI pmp3g-prognos (timvis, ~10 dygn).
    Extraherar relevanta parametrar för solprognos.
    """
    url = (
        f"https://opendata-download-metfcst.smhi.se/api/category/pmp3g"
        f"/version/2/geotype/point/lon/{lon}/lat/{lat}/data.json"
    )
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    data = r.json()

    wanted = {
        't':        'temp_c',
        'r':        'humidity_pct',
        'tcc_mean': 'cloud_total_octas',
        'lcc_mean': 'cloud_low_octas',
        'Wsymb2':   'weather_symbol',
    }
    rows = []
    for step in data['timeSeries']:
        ts  = pd.Timestamp(step['validTime']).tz_convert('Europe/Stockholm')
        row = {'timestamp': ts}
        for p in step['parameters']:
            if p['name'] in wanted:
                row[wanted[p['name']]] = p['values'][0]
        rows.append(row)

    df_fcst = pd.DataFrame(rows).set_index('timestamp')
    # Molntäckning oktanter → procent
    for col in ['cloud_total_octas', 'cloud_low_octas']:
        df_fcst[col.replace('octas', 'pct')] = df_fcst[col] * (100 / 8)
    return df_fcst


def build_tomorrow_features(daily_df, pmp3g_df, strang_yesterday):
    """
    Bygger feature-vektor för imorgon baserat på
    - Historiska laggade värden
    - SMHI pmp3g molntäckning (proxy för STRÅNG)
    """
    tomorrow = (pd.Timestamp.now(tz='Europe/Stockholm') + pd.Timedelta(days=1)).date()
    tmrw_fcst = pmp3g_df[pmp3g_df.index.date == tomorrow]

    # Solproduktiva timmar
    solar = tmrw_fcst.between_time('06:00', '20:00')

    # Molntäckning → approximera GHI-potential
    # Klar himmel ≈ 800 W/m² (sommar) / 400 W/m² (vinter)
    month = tomorrow.month
    clear_sky_max = 200 + 600 * np.sin(np.pi * (month - 1) / 11)
    cloud_factor  = 1 - solar['cloud_total_pct'].mean() / 100
    strang_est    = clear_sky_max * cloud_factor

    last = daily_df.iloc[-1]

    features = {
        'strang_lag1':   strang_yesterday,
        'strang_lag2':   daily_df['strang_mean'].iloc[-2] if len(daily_df)>1 else strang_yesterday,
        'strang_lag7':   daily_df['strang_mean'].iloc[-7] if len(daily_df)>6 else strang_yesterday,
        'strang_roll7':  daily_df['strang_mean'].iloc[-7:].mean(),
        'strang_max':    strang_est * 1.3,
        'temp_max':      tmrw_fcst['temp_c'].max(),
        'temp_mean':     tmrw_fcst['temp_c'].mean(),
        'prod_lag1':     last['production'],
        'prod_lag2':     daily_df['production'].iloc[-2] if len(daily_df)>1 else last['production'],
        'prod_lag7':     daily_df['production'].iloc[-7] if len(daily_df)>6 else last['production'],
        'prod_roll7':    daily_df['production'].iloc[-7:].mean(),
        'prod_roll14':   daily_df['production'].iloc[-14:].mean(),
        'month':         tomorrow.month,
        'weekday':       tomorrow.weekday(),
        'season_sin':    np.sin(2 * np.pi * tomorrow.timetuple().tm_yday / 365.25),
        'season_cos':    np.cos(2 * np.pi * tomorrow.timetuple().tm_yday / 365.25),
    }
    return features, tomorrow


# ── Kör ──
pmp3g_df       = fetch_pmp3g_forecast(LAT, LON)
yesterday_date = datetime.now() - timedelta(days=1)
strang_yest    = fetch_strang_forecast_day(LAT, LON, STRANG_PARAM, yesterday_date)
strang_yest_mean = strang_yest.mean()

features_tomorrow, tomorrow_date = build_tomorrow_features(daily, pmp3g_df, strang_yest_mean)
X_pred = pd.DataFrame([features_tomorrow])[FEATURES]

predicted_kwh = model.predict(X_pred)[0]
predicted_kwh = max(0, predicted_kwh)  # Produktion kan inte vara negativ

print(f"\n📅 Prognos för: {tomorrow_date}")
print(f"🌤  Molntäckning (snitt): {pmp3g_df[pmp3g_df.index.date == tomorrow_date]['cloud_total_pct'].mean():.0f}%")
print(f"🌡  Temperatur max:       {pmp3g_df[pmp3g_df.index.date == tomorrow_date]['temp_c'].max():.1f}°C")
print(f"⚡  Prognos produktion:   {predicted_kwh:.1f} kWh")


## Cell 9 — Batteristyrning & lastflyttning baserad på prognos

In [ ]:
# ── Konstantar (anpassa till din installation) ──
BATTERY_CAP_KWH   = 18.0
BATTERY_EFF       = 0.90
TAX_FEES_KR_KWH   = 0.7075    # Skatter + nätavgift + handel inkl moms
MOMS              = 1.25
SELL_ADJ_KR_KWH   = 0.029

# Flexibla laster per dygn (kWh som KAN styras)
FLEX_LOADS = {
    'elbil':      15.0,    # Schemalagd laddning
    'varmvatten':  3.5,    # Beredaren kan startas/stoppas
    'diskmaskin':  1.2,
    'tvattmaskin': 1.8,
    'torktumlare': 2.5,
}
FLEX_TOTAL_KWH = sum(FLEX_LOADS.values())

def plan_day(predicted_prod_kwh, pmp3g_df, target_date, battery_soc_start=0.0):
    """
    Skapar en timvis styrningsplan för:
    1. Batteriladdning/-urladdning
    2. Lastflyttning av flexibla apparater

    Returnerar DataFrame med rekommendation per timme.
    """
    day_df = pmp3g_df[pmp3g_df.index.date == target_date].copy()
    if day_df.empty:
        print(f"⚠️  Ingen prognos för {target_date}"); return None

    # Spotprisdata saknas i pmp3g → använd typisk dag-profil som proxy
    # (Ersätt med Tibber/Nordpool-API för riktiga priser)
    hour_idx = day_df.index.hour
    spot_proxy = pd.Series(
        0.40 + 0.25 * np.sin(np.pi * (hour_idx - 6) / 18),   # Lågt dagtid, högt morgon/kväll
        index=day_df.index
    ).clip(lower=0.15)
    day_df['spot_kr']   = spot_proxy
    day_df['buy_price'] = day_df['spot_kr'] * MOMS + TAX_FEES_KR_KWH
    day_df['sell_price']= day_df['spot_kr'] + SELL_ADJ_KR_KWH

    # ── Solprofil: fördela prognos över solproduktiva timmar ──
    solar_hours = day_df.between_time('06:00', '20:00').index
    sin_weights = np.sin(np.pi * np.arange(len(solar_hours)) / max(len(solar_hours)-1, 1))
    sin_weights = sin_weights / sin_weights.sum()
    day_df['solar_prod_kwh'] = 0.0
    day_df.loc[solar_hours, 'solar_prod_kwh'] = predicted_prod_kwh * sin_weights

    # ── Lastflyttning: placera flex-laster under sol/billigaste timmar ──
    # Sortera timmar: solöverskott först, sedan lägst pris
    day_df['priority_score'] = (
        day_df['solar_prod_kwh'] * 3          # Solöverskott väger tyngst
        - day_df['buy_price'] * 5             # Lågt pris näst viktigast
        + (1 - day_df['cloud_total_pct']/100) # Klar himmel bonus
    )
    sorted_hours = day_df.sort_values('priority_score', ascending=False)

    day_df['flex_load_kwh'] = 0.0
    remaining_flex = FLEX_TOTAL_KWH
    for ts in sorted_hours.index:
        if remaining_flex <= 0: break
        alloc = min(remaining_flex, FLEX_TOTAL_KWH / 8)   # Max 1/8 per timme
        day_df.loc[ts, 'flex_load_kwh'] = alloc
        remaining_flex -= alloc

    # ── Batterilogik ──
    soc = battery_soc_start
    day_df['soc_kwh']      = 0.0
    day_df['bat_action']   = 'vila'
    day_df['bat_kwh']      = 0.0

    for ts, row in day_df.iterrows():
        net = row['solar_prod_kwh'] - row.get('base_load', 0.8) - row['flex_load_kwh']

        if net > 0:                                        # Överskott → ladda
            charge = min(net, (BATTERY_CAP_KWH - soc) / BATTERY_EFF)
            soc += charge * BATTERY_EFF
            day_df.loc[ts, 'bat_action'] = 'laddar'
            day_df.loc[ts, 'bat_kwh'] = charge
        elif row['buy_price'] > day_df['buy_price'].quantile(0.65) and soc > 2:
            discharge = min(abs(net) if net < 0 else 1.5, soc)
            soc -= discharge
            day_df.loc[ts, 'bat_action'] = 'urladdar'
            day_df.loc[ts, 'bat_kwh'] = discharge

        soc = np.clip(soc, 0, BATTERY_CAP_KWH)
        day_df.loc[ts, 'soc_kwh'] = soc

    return day_df


# ── Kör plan för imorgon ──
plan = plan_day(predicted_kwh, pmp3g_df, tomorrow_date)

if plan is not None:
    print(f"\n📋 STYRNINGSPLAN — {tomorrow_date}")
    print(f"{'Tid':6} {'Sol':>7} {'Flex':>6} {'Batteri':>9} {'SOC':>6} {'Köppris':>8}")
    print('─' * 50)
    for ts, row in plan.iterrows():
        print(
            f"{ts.strftime('%H:00'):6} "
            f"{row['solar_prod_kwh']:>6.2f}k "
            f"{row['flex_load_kwh']:>5.2f}k "
            f"{row['bat_action']:>9} "
            f"{row['soc_kwh']:>5.1f}k "
            f"{row['buy_price']:>7.2f}kr"
        )
    print()
    solar_self = plan['solar_prod_kwh'].clip(upper=plan.get('base_load', 0.8) + plan['flex_load_kwh']).sum()
    print(f"⚡ Prognostiserad produktion:      {predicted_kwh:.1f} kWh")
    print(f"🔋 Max SOC under dygnet:           {plan['soc_kwh'].max():.1f} kWh")
    print(f"🏠 Flexibel last styrd:             {plan['flex_load_kwh'].sum():.1f} kWh")
    print(f"💰 Uppskattad köpt el (utan flex):  {(plan['solar_prod_kwh']==0).sum() * 0.8:.1f} kWh")
